In [ ]:
import os
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings


# Ollama embedding model
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [3]:
docs = [
    Document(page_content="Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.",
             metadata={"topic": "space"}),
    Document(page_content="The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.",
             metadata={"topic": "space"}),
    Document(page_content="Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.",
             metadata={"topic": "space"}),
    Document(page_content="NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.",
             metadata={"topic": "space"}),
    Document(page_content="Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.",
             metadata={"topic": "space"}),
    Document(page_content="DNA is a double-helix molecule that carries the genetic instructions for all living organisms.",
             metadata={"topic": "biology"}),
    Document(page_content="Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.",
             metadata={"topic": "biology"}),
    Document(page_content="The Roman Empire at its peak covered over 5 million square kilometers across three continents.",
             metadata={"topic": "history"}),
    Document(page_content="The printing press, invented by Gutenberg around 1440, revolutionised the spread of knowledge.",
             metadata={"topic": "history"}),
    Document(page_content="The Amazon River discharges more freshwater into the ocean than any other river on Earth.",
             metadata={"topic": "geography"}),
    Document(page_content="The Sahara Desert spans about 9.2 million square kilometers across northern Africa.",
             metadata={"topic": "geography"}),
    Document(page_content="Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.",
             metadata={"topic": "biology"}),
]

In [4]:
for ind, doc in enumerate(docs,1):
    print(f"Doc No.: {ind} | Topic: {doc.metadata["topic"]}")
    print(f"Content: {doc.page_content}\n")

Doc No.: 1 | Topic: space
Content: Rockets work by expelling gas at high speed, generating thrust through Newton's third law of motion.

Doc No.: 2 | Topic: space
Content: The International Space Station orbits Earth at about 400 km altitude and travels at 28,000 km/h.

Doc No.: 3 | Topic: space
Content: Spacecraft use gravitational slingshots around planets to gain speed without burning extra fuel.

Doc No.: 4 | Topic: space
Content: NASA's Voyager 1 is the farthest human-made object, now over 23 billion km from the Sun.

Doc No.: 5 | Topic: space
Content: Solar sails use radiation pressure from sunlight to slowly propel spacecraft without fuel.

Doc No.: 6 | Topic: biology
Content: DNA is a double-helix molecule that carries the genetic instructions for all living organisms.

Doc No.: 7 | Topic: biology
Content: Photosynthesis allows plants to convert sunlight, water, and CO2 into glucose and oxygen.

Doc No.: 8 | Topic: history
Content: The Roman Empire at its peak covered over 5 mi

In [8]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="similarity_search_demo",
)
vectorstore.get()


{'ids': ['f16b6c98-53d6-4526-9ab5-8669b6776552',
  '0e9be109-ae2c-416f-b9e6-7fa43c8a395b',
  'ee8b6d31-7794-45fa-bc36-c5c40e6e80b8',
  '4f6601b7-fac4-4da0-9ff9-cb3c2538f01c',
  '95d53083-2a6a-4710-869f-a7b741eb12e3',
  '8ef04321-ab73-436f-9b85-0765c3ad4d88',
  '528170ef-31e5-4790-a6ae-7aa3b014fcf9',
  'fdc11efb-769b-48b9-8ad1-6b52f0dbe02d',
  '7e5007b5-63e3-4720-88a5-d968f8269968',
  'dcee51d8-d350-44d6-93f2-e8862bdf04a6',
  '5fb66d3c-3fa5-4320-bc98-7262295ed997',
  '047d9629-29db-440e-90e2-39cd7db5bcf5',
  'fd173c4b-e169-496c-aab4-49b8ad5ca302',
  'd1d4ef0d-7827-4054-b978-91c151e836e4',
  '41a83911-5851-4275-871b-0d4d5003bf60',
  '478bbb16-4ca3-4718-ba6d-677573a6d200',
  '82c51909-2b11-4a71-9f48-3052cda83913',
  '778a897e-74ca-4311-b164-8cb13431b5f3',
  '61def326-f1e7-4cf8-8679-3f36343ec6aa',
  '83baf2f2-ec2d-4277-87dd-3c1b4f22d255',
  'd057a43f-0e62-4860-9aa8-2ff6f6c668bb',
  '21b75a64-0734-4d39-b26f-573c65613e0f',
  'acd39d71-847e-44d5-b6ac-b8a38b7ed999',
  'c074c18c-c6ad-4967-9f39-

In [14]:
retriever = vectorstore.as_retriever(
    search_type="similarity",  # similarity, similarity_score_threshold, mmr
    search_kwargs={"k": 2, "filter" : {"topic": "biology"} },   # k is the number of retrieved docs
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x7d558851b4d0>, search_kwargs={'k': 2, 'filter': {'topic': 'biology'}})

In [16]:
# this method is not used generally inside chains, or if you require customizations then 
# also you do not use this method
query = "How do cells generate their energy?"

results = vectorstore.similarity_search(query, k=3)

print(f"{query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i} [topic={doc.metadata['topic']}]")
    print(f"{doc.page_content}")
    print()

How do cells generate their energy?

Result 1 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 2 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 3 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.



In [17]:
query = "How do rockets work?"

# retrievers in langchain are runnables, create chains using retrievers or invoke
results = retriever.invoke(query)

# All returned documents should be from the space topic
print(f"{query}\n")
for i, doc in enumerate(results, 1):
    print(f"Result {i} [topic={doc.metadata['topic']}]")
    print(f"{doc.page_content}")
    print()

How do rockets work?

Result 1 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

Result 2 [topic=biology]
Mitochondria generate ATP through cellular respiration, powering nearly all cellular processes.

